In [3]:
from pathlib import Path
import pandas as pd

SOURCE_CSV = Path("../data/csv/btcusd_bitstamp_1min_2012-2025.csv.gz")
OUT_CSV = Path("../data/raw/btc_5m_ohlcv.csv")

df = pd.read_csv(SOURCE_CSV, compression="gzip")
df.columns = [c.strip().lower() for c in df.columns]

df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

for col in ["open", "high", "low", "close", "volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = (
    df.dropna(subset=["timestamp", "open", "high", "low", "close", "volume"])
      .drop_duplicates(subset=["timestamp"], keep="last")
      .sort_values("timestamp")
      .set_index("timestamp")
)

df_5m = (
    df.resample("5min", label="left", closed="left")
      .agg({
          "open": "first",
          "high": "max",
          "low": "min",
          "close": "last",
          "volume": "sum"
      })
      .dropna(subset=["open", "high", "low", "close"])
      .reset_index()
)

df_5m["timestamp"] = df_5m["timestamp"].dt.strftime("%Y-%m-%d %H:%M")

df_5m = df_5m[["timestamp", "open", "high", "low", "close", "volume"]]
df_5m.to_csv(OUT_CSV, index=False)

print(df_5m.head())
print(df_5m.tail())
print(f"Saved to: {OUT_CSV.resolve()}")

          timestamp  open  high   low  close  volume
0  2012-01-01 10:00  4.58  4.58  4.58   4.58     0.0
1  2012-01-01 10:05  4.58  4.58  4.58   4.58     0.0
2  2012-01-01 10:10  4.58  4.58  4.58   4.58     0.0
3  2012-01-01 10:15  4.58  4.58  4.58   4.58     0.0
4  2012-01-01 10:20  4.58  4.58  4.58   4.58     0.0
                timestamp      open      high       low     close    volume
1369316  2025-01-06 23:40  102149.0  102259.0  102119.0  102225.0  0.360514
1369317  2025-01-06 23:45  102223.0  102262.0  102201.0  102235.0  0.474493
1369318  2025-01-06 23:50  102237.0  102246.0  102213.0  102227.0  0.796058
1369319  2025-01-06 23:55  102227.0  102280.0  102225.0  102280.0  0.499571
1369320  2025-01-07 00:00  102278.0  102291.0  102263.0  102263.0  0.523107
Saved to: C:\btc-volatility-forecast\btc-volatility-forecast\data\raw\btc_5m_ohlcv.csv
